# الداتا بعد الديب كليننق

# الداتا

In [ ]:
import pandas as pd
from datasets import Dataset

# 1) مسار الملف النظيف
csv_path = r"C:\Users\jorif\Downloads\nlp_data\nlp_data\clean_jory.csv"

# 2) قراءة الداتا
df = pd.read_csv(csv_path, encoding="utf-8")

print("Shape:", df.shape)
print(df.columns)

# نتأكد ما فيه NaN بعد التنظيف (احتياط)
df = df.dropna(subset=["text_clean", "summary_clean"])

# نخلي الأسماء مناسبة لـ HuggingFace Dataset
df_hf = df[["text_clean", "summary_clean"]].rename(
    columns={
        "text_clean": "text",
        "summary_clean": "summary"
    }
)

# 3) تحويل إلى HF Dataset
dataset = Dataset.from_pandas(df_hf, preserve_index=False)

print(dataset)
print(dataset[0])


c:\Users\jorif\anaconda3\envs\nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Shape: (4985, 14)
Index(['title', 'description', 'date_download', 'date_publish',
       'source_domain', 'url', 'text', 'image_url', 'authors', 'title_page',
       'word_count', 'summary_3points', 'text_clean', 'summary_clean'],
      dtype='object')
Dataset({
    features: ['text', 'summary'],
    num_rows: 4985
})
{'text': 'فاز القاضي بريت كافانو مرشح الرئيس الأمريكي دونالد ترامب إلى المحكمة العليا، الجمعة، بتصويت أولي إجرائي في مجلس الشيوخ نحو تثبيته في المنصب عبر تصويت نهائي السبت. ونال القاضي بريت كافانو 51 صوتا مقابل معارضة 49. وكانت السيناتورة الجمهورية ليزا موركوسكي صوتت ضد تثبيت ترشيح كافانو للمحكمة العليا في هذا الاقتراع الأولي، الذي يعتبر مؤشرا للتصويت النهائي الذي سيجرى السبت. وصوت العضوان الجمهوريان المعتدلان جيف فلايك وسوزان كولينز لمصلحة كافانو، وكذلك فعل السيناتور الديمقراطي جو مانتشن. وسارع ترامب إلى الترحيب بقرار الكونغرس الأمريكي، معلنا عبر تويتر أنه "فخور جدا" بقرار مجلس الشيوخ المضي قدما في إجراء تصويت نهائي على تثبيت كافانو في المحكمة العليا. وقال ترامب أنا "فخو

In [ ]:

# تحميل AraT5 من المجلد المحلي + نقل الـ model إلى GPU
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


model_path = r"C:\Users\jorif\Downloads\nlp_data\nlp_data\AraT5_summarizer"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# لو فيه CUDA حنستخدم الـ GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Model is on:", model.device)


Model is on: cpu


In [ ]:


#دالة الـ preprocessing (توكنايز) مع max_length اللي اتفقنا عليه
from transformers import DataCollatorForSeq2Seq

max_input_length = 750
max_target_length = 150

def preprocess_function(examples):
    inputs = examples["text"]
    targets = examples["summary"]

    # ترميز الخبر (input)
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    # ترميز الملخص (labels)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=max_target_length,
            truncation=True,
            padding="max_length"
        )["input_ids"]

    model_inputs["labels"] = labels
    return model_inputs

# نطبّق الـ preprocess على كامل الداتا (كلها train)
tokenized_train = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset.column_names  # نشيل الأعمدة الأصلية ونخلي بس الـ ids
)

print(tokenized_train)


Map:   0%|          | 0/4985 [00:00<?, ? examples/s]c:\Users\jorif\anaconda3\envs\nlp\lib\site-packages\transformers\tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 4985/4985 [00:04<00:00, 1132.18 examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4985
})


In [ ]:
#عداد الـ DataCollator و TrainingArguments و Trainer
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)
training_args = Seq2SeqTrainingArguments(
    output_dir="./arat5_GeminiNews_clean_run1",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    optim="adafactor",

    # ما عندنا validation
    eval_strategy="no",       # ← المتغير الصحيح الآن
    save_strategy="epoch",

    predict_with_generate=True,

    logging_steps=100,

    bf16=False,
    fp16=False,

    max_grad_norm=5.0,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=None,          # ما عندنا eval
    data_collator=data_collator,
    tokenizer=tokenizer,
)


C:\Users\jorif\AppData\Local\Temp\ipykernel_8068\4269988227.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

# حفظ المودل + التوكنايزر في نفس المجلد
trainer.save_model("./arat5_GeminiNews_clean_run1")
tokenizer.save_pretrained("./arat5_GeminiNews_clean_run1")

print("✅ Training finished & model saved!")


c:\Users\jorif\anaconda3\envs\nlp\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
100,36.409300
200,17.960100
300,8.314000
400,6.265700
500,3.382400
600,1.573800
700,1.264200
800,1.132600
900,1.083900
1000,1.076600


c:\Users\jorif\anaconda3\envs\nlp\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\jorif\anaconda3\envs\nlp\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


✅ Training finished & model saved!


In [ ]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# ============================================
# 1) تحميل المودل والتوكنايزر من مجلد التدريب
# ============================================

model_path = r"C:\Users\jorif\Downloads\nlp_data\nlp_data\arat5_GeminiNews_clean_run1\checkpoint-3741"   # غيّريه لو اسم مجلدك مختلف

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Model loaded on:", device)

# ============================================
# 2) دالة التلخيص
# ============================================
def summarize(text, max_input_length=750, max_target_length=150):

    inputs = tokenizer(
        text,
        max_length=max_input_length,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    ).to(device)

    # نولّد الملخص
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=max_target_length,
            num_beams=4,
            early_stopping=True
        )

    summary = tokenizer.decode(output[0], skip_special_tokens=True)
    return summary


# ============================================
# 3) الخبر الذي تريدين اختباره
# ============================================
test_news = """
أعلنت وزارة الصحة في المملكة العربية السعودية اليوم عن إطلاق مبادرة وطنية جديدة تهدف إلى تحسين جودة الخدمات الصحية في المستشفيات الحكومية عبر استخدام تقنيات الذكاء الاصطناعي. وذكرت الوزارة أن المشروع يأتي ضمن مساعيها لتحقيق مستهدفات رؤية 2030، التي تركّز على تطوير القطاع الصحي ورفع كفاءة العمل الطبي، مشيرة إلى أن التقنية الجديدة ستُستخدم لتحليل بيانات المرضى بشكل أكثر دقة وسرعة، مما يساهم في تسريع عملية التشخيص وتقليل الأخطاء الطبية.

وأوضحت الوزارة أن النظام الجديد سيعتمد على خوارزميات متقدمة قادرة على التعرّف على أنماط الأمراض المزمنة ومتابعة حالات المرضى عبر الزمن، كما يمكنه تنبيه الأطباء في حال حدوث تغيّر مفاجئ في المؤشرات الحيوية للمريض. وأضاف البيان أن هذه التقنيات ستساعد على تخفيف الضغط على الكادر الطبي، خصوصًا في أقسام الطوارئ والعناية المركزة التي تعاني من كثافة العمل.

وأكدت وزارة الصحة أنها بدأت بالفعل في المرحلة الأولى من المشروع عبر تركيب الأنظمة في خمس مستشفيات رئيسية في الرياض وجدة والدمام، مع خطط للتوسع تدريجيًا خلال العام المقبل. كما أعلنت الوزارة أن المشروع يشمل تدريب أكثر من 1500 موظف طبي وإداري على استخدام الأنظمة الجديدة، إضافة إلى تهيئة البنية التحتية الرقمية اللازمة لتشغيل النظام بكفاءة عالية.

وختمت الوزارة بيانها بالتأكيد على أن هذه الخطوة تمثل نقلة مهمة في مسيرة التحول الرقمي للقطاع الصحي في المملكة، وأنها ستسهم في تقديم رعاية صحية أفضل للمواطنين والمقيمين، وتوفير بيئة طبية أكثر أمانًا ودقة وانتظامًا.
"""

# ============================================
# 4) التلخيص
# ============================================
summary = summarize(test_news)

print("===== الملخص =====\n")
print(summary)


Model loaded on: cpu
===== الملخص =====

1. أطلقت وزارة الصحة السعودية مبادرة وطنية جديدة لتحسين جودة الخدمات الصحية في المستشفيات الحكومية باستخدام تقنيات الذكاء الاصطناعي. 2. تهدف المبادرة إلى تحسين جودة الخدمات الصحية في المستشفيات الحكومية عبر تقنيات الذكاء الاصطناعي. 3. تشمل المرحلة الأولى تركيب الأنظمة وتدريب أكثر من 1500 موظف طبي وإداري.


In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import re

# 1) Load fine-tuned AraT5 model from local folder
model_path = r"C:\Users\jorif\Downloads\nlp_data\nlp_data\arat5_GeminiNews_clean_run1\checkpoint-3741"

tokenizer = AutoTokenizer.from_pretrained(model_path)

# ================================
# 🆕 select device automatically
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 🆕 load model on the device
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
print("Model loaded on:", model.device)


# 2) Small helper: remove repeated tail like "... تقليل الأخطاء الطبية وتقليل الأخطاء الطبية"
def clean_repeated_tail(sentence: str) -> str:
    words = sentence.split()
    for k in range(3, len(words) // 2 + 1):
        tail = words[-k:]
        prev = words[-2 * k : -k]
        if tail == prev and tail:
            words = words[:-k]
            break
    return " ".join(words)


# 3) حذف النقاط المكررة
def smart_deduplicate(points, min_len: int = 10):
    cleaned = []
    for p in points:
        p_stripped = p.strip()
        if len(p_stripped) < min_len:
            continue
        if any(p_stripped == q for q in cleaned):
            continue
        if any(p_stripped in q or q in p_stripped for q in cleaned):
            continue
        cleaned.append(p_stripped)
    return cleaned[:3]


# 4) تحويل مخرجات المودل إلى نقاط 1 / 2 / 3
def parse_points(summary_text: str) -> str:
    text = summary_text.replace("\n", " ")
    parts = re.split(r"\s*[1-3]\s*[\.\-،:]\s*", text)
    raw_points = [p for p in parts[1:] if p.strip()]
    raw_points = [clean_repeated_tail(p) for p in raw_points]
    points = smart_deduplicate(raw_points)
    numbered = [f"{i+1}. {p}" for i, p in enumerate(points)]
    return "\n".join(numbered)


# 5) Main summarization function
def summarize(text, max_input_length=750, max_target_length=150):
    enc = tokenizer(
        text,
        max_length=max_input_length,
        truncation=True,
        padding=False,
        return_tensors="pt"
    )
    # 🆕 send input_ids to same device
    input_ids = enc.input_ids.to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_length=max_target_length,
            num_beams=4,
            no_repeat_ngram_size=4,
            repetition_penalty=1.2,
            length_penalty=0.8,
            early_stopping=True
        )

    raw_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    cleaned_summary = parse_points(raw_summary)
    return raw_summary, cleaned_summary


# 6) test
test_text = """
أعلنت وزارة الصحة في المملكة العربية السعودية اليوم عن إطلاق مبادرة وطنية جديدة تهدف إلى تحسين جودة الخدمات الصحية في المستشفيات الحكومية عبر استخدام تقنيات الذكاء الاصطناعي...
"""

raw, cleaned = summarize(test_text)

print("===== RAW MODEL OUTPUT =====")
print(raw)
print("\n===== CLEANED SUMMARY =====")
print(cleaned)


Using device: cpu
Model loaded on: cpu
===== RAW MODEL OUTPUT =====
1. أطلقت وزارة الصحة السعودية مبادرة وطنية جديدة لتحسين جودة الخدمات الصحية بالمستشفيات الحكومية. 2. تهدف المبادرة إلى تحسين جودة الخدمات الصحية باستخدام الذكاء الاصطناعي. 3. تهدف المبادرة لتحسين جودة الخدمات الطبية في المستشفيات الحكومية باستخدام تقنيات الذكاء الإصطناعي.

===== CLEANED SUMMARY =====
1. أطلقت وزارة الصحة السعودية مبادرة وطنية جديدة لتحسين جودة الخدمات الصحية بالمستشفيات الحكومية.
2. تهدف المبادرة إلى تحسين جودة الخدمات الصحية باستخدام الذكاء الاصطناعي.
3. تهدف المبادرة لتحسين جودة الخدمات الطبية في المستشفيات الحكومية باستخدام تقنيات الذكاء الإصطناعي.
